# Conflict Panel — Construction

Builds `conflict_panel.csv`: a balanced monthly panel of 201 state-based conflicts (UCDP GED v25.1, January 1989–December 2024) merged with PA-X v9 peace agreements. Each section handles one construction step: expanding GED events to monthly resolution, building the balanced panel, merging agreements, and computing timing and treatment variables. For variable definitions and design choices, see the codebook (`codebook_conflict_panel.md`) and the technical notes (`reports/conflict_panel_construction.tex`).

In [49]:
import pandas as pd
import numpy as np

---
## 1. UCDP GED — expand events to monthly resolution

In [50]:
df_ucdp = pd.read_csv('../../data/input/ucdp/GEDEvent_v25_1.csv', low_memory=False)
df_ucdp = df_ucdp[df_ucdp['type_of_violence'] == 1].reset_index(drop=True).copy()

relevant_columns = [
    'country', 'conflict_new_id', 'date_start', 'date_end', 'best',
    'dyad_new_id', 'type_of_violence', 'region', 'conflict_name', 'side_a', 'side_b'
]
df_ucdp = df_ucdp[relevant_columns].sort_values(['country', 'conflict_new_id', 'date_start'])

isocodes = pd.read_csv('../../data/input/isocodes/isocodes_appended.csv', sep=';')
df_ucdp = df_ucdp.merge(
    isocodes[['name', 'alpha_3']], left_on='country', right_on='name', how='left'
).rename(columns={'alpha_3': 'isocode'})

print(f"GED events: {len(df_ucdp):,} rows, {df_ucdp['conflict_new_id'].nunique()} conflicts")

GED events: 271,331 rows, 201 conflicts


In [51]:
# Each GED event may span multiple months — expand to one row per month,
# distributing fatalities uniformly across the event's duration.
df_ucdp["date_start"] = pd.to_datetime(df_ucdp["date_start"]).dt.to_period("M").dt.to_timestamp()
df_ucdp["date_end"]   = pd.to_datetime(df_ucdp["date_end"]).dt.to_period("M").dt.to_timestamp()

n_months = (
    (df_ucdp["date_end"].dt.to_period("M") - df_ucdp["date_start"].dt.to_period("M"))
    .apply(lambda x: x.n + 1)
)

df_expanded = df_ucdp.loc[df_ucdp.index.repeat(n_months)].copy()
month_offsets = df_expanded.groupby(level=0).cumcount()
df_expanded["date"] = (
    df_expanded["date_start"].dt.to_period("M")
    .add(month_offsets.values)
    .dt.to_timestamp()
)
df_expanded["best"] = df_expanded["best"] / n_months.repeat(n_months).values

print(f"Expanded: {len(df_expanded):,} event-month rows")

Expanded: 279,490 event-month rows


In [52]:
# Aggregate to conflict_id × month.
# Result: one row per (conflict, month) where at least one GED event occurred.
df_conflict_month = (
    df_expanded
    .groupby(['conflict_new_id', 'date'], as_index=False)
    .agg(
        best=('best', 'sum'),
        n_events=('dyad_new_id', 'count'),
        dyad_id=('dyad_new_id', 'unique'),
        isocode=('isocode', 'first'),
        country=('country', 'first'),
        region=('region', 'first'),
        n_isocode=('isocode', 'nunique'),
        isocode_array=('isocode', 'unique'),
        countries_array=('country', 'unique'),
        type_of_violence=('type_of_violence', 'min')
    )
)

# Dominant country: the one with the highest total fatalities across all events
main_isocode = (
    df_expanded.groupby(['conflict_new_id', 'isocode', 'country', 'region'])
    .agg(sum_best=('best', 'sum'))
    .reset_index()
    .sort_values(['conflict_new_id', 'sum_best'], ascending=[True, False])
    .drop_duplicates(['conflict_new_id'])
)
df_conflict_month = df_conflict_month.merge(
    main_isocode[['conflict_new_id', 'isocode', 'country', 'region']],
    on=['conflict_new_id'], how='left'
)
df_conflict_month = df_conflict_month.drop(columns=['isocode_x', 'country_x', 'region_x'])
df_conflict_month = df_conflict_month.rename(columns={
    'isocode_y': 'isocode', 'country_y': 'country', 'region_y': 'region',
    'conflict_new_id': 'conflict_id', 'date': 'year_mo'
})
df_conflict_month['best'] = df_conflict_month['best'].fillna(0.0)

print(f"df_conflict_month: {df_conflict_month['conflict_id'].nunique()} conflicts, "
      f"{len(df_conflict_month):,} event-months")

df_conflict_month: 201 conflicts, 16,053 event-months


---
## 2. PA-X Agreements

In [37]:
df_pax = pd.read_csv('../../data/input/pax/pax_data_2144_agreements_v9_10.csv', low_memory=False)
df_pax.columns = df_pax.columns.str.lower()
df_pax = df_pax.rename(columns={"ucdpcon": "conflict_id", "loc1iso": "isocode"})

df_pax["dat"] = pd.to_datetime(df_pax["dat"])
df_pax["year_mo"] = df_pax["dat"].dt.to_period("M").dt.to_timestamp()

# SSD → SDN before South Sudan independence (July 2011)
mask = (df_pax["isocode"] == "SSD") & (
    (df_pax["dat"].dt.year < 2011) |
    ((df_pax["dat"].dt.year == 2011) & (df_pax["dat"].dt.month <= 6))
)
df_pax.loc[mask, "isocode"] = "SDN"

df_pax = df_pax.dropna(subset=['conflict_id', 'year_mo'])
df_pax["conflict_id"] = df_pax["conflict_id"].astype(int)

# Agreement stage indicators
df_pax["agreement"]            = 1
df_pax["subs_agreement"]       = (df_pax["stage"].str.lower() == "subpar").astype(int)
df_pax["comp_agreement"]       = (df_pax["stage"].str.lower() == "subcomp").astype(int)
df_pax["cea_agreement"]        = (df_pax["stage"].str.lower() == "cea").astype(int)
df_pax["cea_ceamix_agreement"] = (df_pax["stagesub"].str.lower() == "ceamix").astype(int)
df_pax["cea_ceas_agreement"]   = (df_pax["stagesub"].str.lower() == "ceas").astype(int)
df_pax["cea_rel_agreement"]    = (df_pax["stagesub"].str.lower() == "rel").astype(int)

# PA-X provision columns (used for heterogeneous treatment effects in CSDID)
features_cluster_columns = [
    'hriimon', 'ime', 'ddrprog', 'imun', 'tjrep', 'ppsaut', 'tjmis',
    'protgrp', 'ppsvet', 'hriibod', 'tpsloc', 'terps', 'pol', 'epsres',
    'impk', 'epsoth', 'polnewtemp', 'protciv', 'hrtrinc', 'ssrddr',
    'ceprov', 'ppsint', 'ddrdemil', 'polpar', 'polps', 'hrbor', 'hrdem',
    'medlog', 'tjvet', 'medsubs', 'epsfis', 'tpssub', 'tpsoth', 'eleccomm',
    'eps', 'ssrpol', 'medgov', 'imref', 'tpsaut', 'med', 'hrii', 'cegen', 'ssrgua',
    'ssrarm', 'tjcou', 'ppsoro', 'tjamban', 'tjvic', 'tjmech', 'polnewind', 'mpsme',
    'mpsjt', 'imoth', 'ppsothpr', 'ppsex', 'hrmob', 'ele', 'hrni', 'civso', 'pubad',
    'juscr', 'mps', 'prot', 'mpspro'
]

agreement_type_cols = [
    'agreement', 'comp_agreement', 'subs_agreement', 'cea_agreement',
    'cea_ceamix_agreement', 'cea_ceas_agreement', 'cea_rel_agreement', 'ce'
]

df_agreements = (
    df_pax
    .groupby(["conflict_id", "year_mo"], as_index=False)
    .agg(
        agreement=("agreement", "max"),
        comp_agreement=("comp_agreement", "max"),
        subs_agreement=("subs_agreement", "max"),
        cea_agreement=("cea_agreement", "max"),
        cea_ceamix_agreement=("cea_ceamix_agreement", "max"),
        cea_ceas_agreement=("cea_ceas_agreement", "max"),
        cea_rel_agreement=("cea_rel_agreement", "max"),
        ce=("ce", "max"),
        **{col: (col, "max") for col in features_cluster_columns}
    )
)

# Restrict to conflicts in the GED sample
ged_conflict_ids = set(df_conflict_month['conflict_id'].unique())
df_agreements = df_agreements[df_agreements['conflict_id'].isin(ged_conflict_ids)].copy()

print(f"PA-X agreement-months linked to GED conflicts: {len(df_agreements):,}")
print(f"Conflicts with at least one PA-X agreement: {df_agreements['conflict_id'].nunique()}")

PA-X agreement-months linked to GED conflicts: 1,271
Conflicts with at least one PA-X agreement: 104


---
## 3. Spell dates per conflict

- `start_date` — first month with any GED event  
- `ged_end_date` — last month with any GED event  
- `end_date` — same as `ged_end_date` for all conflicts  
- `pax_first_date` — first PA-X agreement month (kept for reference)

`end_date` is **not** extended for the 11 conflicts whose first PA-X agreement
arrives after their last GED event. Those agreements are still captured because
the balanced panel in step 4 covers the full global span (so those months exist
and receive a PA-X agreement flag in step 6).

In [38]:
ged_dates = (
    df_conflict_month
    .groupby('conflict_id')['year_mo']
    .agg(start_date='min', ged_end_date='max')
    .reset_index()
)

pax_first = (
    df_agreements
    .groupby('conflict_id')['year_mo']
    .min()
    .reset_index()
    .rename(columns={'year_mo': 'pax_first_date'})
)

conflict_dates = ged_dates.merge(pax_first, on='conflict_id', how='left')

# end_date = ged_end_date for all conflicts (no extension).
# Post-GED agreements are captured via the global panel span in step 4.
conflict_dates['end_date'] = conflict_dates['ged_end_date']

# Diagnostic: conflicts whose first PA-X agreement is after their last GED event
post_ged = conflict_dates[conflict_dates['pax_first_date'] > conflict_dates['ged_end_date']]
print(f"Conflicts whose first PA-X agreement is after the last GED event: {len(post_ged)}")
print("(Agreements captured via global panel — no end_date extension needed)")
print(post_ged[['conflict_id', 'ged_end_date', 'pax_first_date']].to_string(index=False))

Conflicts whose first PA-X agreement is after the last GED event: 11
(Agreements captured via global panel — no end_date extension needed)
 conflict_id ged_end_date pax_first_date
         260   1990-10-01     1991-05-01
         271   1996-09-01     2009-06-01
         298   1989-05-01     1991-05-01
         307   2001-07-01     2010-01-01
         371   1991-03-01     1991-04-01
         407   1997-09-01     1997-12-01
         411   1998-09-01     1999-12-01
         420   2003-04-01     2003-11-01
         425   2009-04-01     2016-07-01
         435   2008-06-01     2010-06-01
       11475   2011-09-01     2012-04-01


---
## 4. Balanced panel — global span

Each conflict gets a continuous monthly series from the **global start** (earliest GED event
across all conflicts) to the **global end** (latest GED event). This matches
`conflict_panel.ipynb` and ensures:

- Dormancy months within each conflict's spell have rows with `best = 0`
- Months after a conflict's last GED event are present in the panel, so the PAX merge
  in step 6 correctly captures the 11 post-GED agreements (without extending `end_date`)

In [39]:
# Global span: same range for all conflicts, matching conflict_panel.ipynb
global_start = df_expanded['date'].min()
global_end   = df_expanded['date'].max()

full_conflict_month = []
for conflict_id in df_conflict_month['conflict_id'].unique():
    months = pd.date_range(global_start, global_end, freq='MS')
    full_conflict_month.append(
        pd.DataFrame({'conflict_id': int(conflict_id), 'year_mo': months})
    )

full_panel = pd.concat(full_conflict_month, ignore_index=True)
print(f"Balanced panel: {full_panel['conflict_id'].nunique()} conflicts, "
      f"{len(full_panel):,} conflict-months")
print(f"Global span: {global_start.strftime('%Y-%m')} to {global_end.strftime('%Y-%m')}")

Balanced panel: 201 conflicts, 86,832 conflict-months
Global span: 1989-01 to 2024-12


---
## 5. Merge GED event data into the balanced panel

`ged_event_month = 1` marks months where UCDP GED recorded at least one event.
All other months have `ged_event_month = 0` and `best = 0` — these are dormancy gaps.

In [40]:
df_panel = full_panel.merge(df_conflict_month, on=['conflict_id', 'year_mo'], how='left')
df_panel = df_panel.sort_values(['conflict_id', 'year_mo']).reset_index(drop=True)

# ged_event_month: 1 if UCDP GED recorded at least one event this month, 0 otherwise
df_panel['ged_event_month'] = df_panel['best'].notna().astype(int)

# Fill 0s for months without GED events
df_panel[['best', 'n_events', 'n_isocode']] = df_panel[['best', 'n_events', 'n_isocode']].fillna(0)
df_panel['log_best'] = np.log1p(df_panel['best'])

# Forward/back fill categorical/meta columns within each conflict
meta_cols = ['isocode', 'country', 'region', 'isocode_array', 'countries_array',
             'type_of_violence', 'dyad_id']
df_panel[meta_cols] = (
    df_panel.groupby('conflict_id')[meta_cols]
    .transform(lambda s: s.ffill().bfill())
)
df_panel['isocode_num'] = df_panel['isocode'].astype('category').cat.codes + 1
df_panel['region_num']  = df_panel['region'].astype('category').cat.codes + 1

print(f"Panel: {len(df_panel):,} rows")
print(f"  GED-event months  (ged_event_month=1): {df_panel['ged_event_month'].sum():,} "
      f"({df_panel['ged_event_month'].mean():.1%})")
print(f"  Dormancy months   (ged_event_month=0): {(df_panel['ged_event_month']==0).sum():,} "
      f"({1 - df_panel['ged_event_month'].mean():.1%})")

Panel: 86,832 rows
  GED-event months  (ged_event_month=1): 16,053 (18.5%)
  Dormancy months   (ged_event_month=0): 70,779 (81.5%)


---
## 6. Merge PA-X agreements onto the full balanced panel

**Critical design choice:** PA-X is merged here, *after* the balanced panel exists.
This captures agreements in:
- Dormancy months (`ged_event_month = 0`) within the conflict spell
- Months after the last GED event (for the 11 conflicts where `end_date` was extended)

Then `agreement_ged` is derived as the intersection of `agreement` and `ged_event_month`,
recovering the original CSDID-appropriate definition without duplicating any code.

In [41]:
df_panel = df_panel.merge(df_agreements, on=['conflict_id', 'year_mo'], how='left')

# Fill 0s for months with no PA-X agreement
all_agreement_cols = agreement_type_cols + features_cluster_columns
df_panel[all_agreement_cols] = df_panel[all_agreement_cols].fillna(0).astype(int)

# ── agreement_ged: agreement that coincides with a GED-recorded event month ──
# This is the treatment indicator used in the CSDID analysis.
# An agreement in a dormancy month cannot reduce fatalities (there are none),
# so CSDID restricts treatment to months with active fighting.
agreement_base_cols = [
    'agreement', 'comp_agreement', 'subs_agreement', 'cea_agreement',
    'cea_ceamix_agreement', 'cea_ceas_agreement', 'cea_rel_agreement'
]
for col in agreement_base_cols:
    df_panel[f'{col}_ged'] = (df_panel[col] == 1) & (df_panel['ged_event_month'] == 1)
    df_panel[f'{col}_ged'] = df_panel[f'{col}_ged'].astype(int)

print("Agreement columns created:")
print(f"  agreement     (any month)          : "
      f"{(df_panel['agreement']==1).sum()} conflict-months, "
      f"{df_panel.groupby('conflict_id')['agreement'].max().sum():.0f} conflicts")
print(f"  agreement_ged (GED-event month only): "
      f"{(df_panel['agreement_ged']==1).sum()} conflict-months, "
      f"{df_panel.groupby('conflict_id')['agreement_ged'].max().sum():.0f} conflicts")

Agreement columns created:
  agreement     (any month)          : 1271 conflict-months, 104 conflicts
  agreement_ged (GED-event month only): 746 conflict-months, 71 conflicts


---
## 7. Add spell dates to panel

In [42]:
df_panel = df_panel.merge(
    conflict_dates[['conflict_id', 'start_date', 'ged_end_date', 'pax_first_date', 'end_date']],
    on='conflict_id', how='left'
)

---
## 8. Timing variables

In [43]:
df_panel['year_mo']      = pd.to_datetime(df_panel['year_mo']).dt.to_period('M')
df_panel['start_date']   = pd.to_datetime(df_panel['start_date']).dt.to_period('M')
df_panel['ged_end_date'] = pd.to_datetime(df_panel['ged_end_date']).dt.to_period('M')
df_panel['end_date']     = pd.to_datetime(df_panel['end_date']).dt.to_period('M')

base_date = df_panel['year_mo'].min()
df_panel['year_mo_numeric']    = (df_panel['year_mo'] - base_date).apply(lambda x: x.n + 1)
df_panel['start_date_numeric'] = (df_panel['start_date'] - base_date).apply(lambda x: x.n + 1)
df_panel['end_date_numeric']   = (df_panel['end_date'] - base_date).apply(lambda x: x.n + 1)

# Conflict age: months elapsed since the conflict's first GED event
df_panel['conflict_age'] = df_panel['year_mo_numeric'] - df_panel['start_date_numeric']
df_panel['conflict_age'] = df_panel['conflict_age'].where(df_panel['conflict_age'] >= 0)

# Active conflict age: cumulative count of months with GED events (ged_event_month = 1)
df_panel['active_conflict_age'] = (
    df_panel.groupby('conflict_id')['ged_event_month'].cumsum() - 1
)
df_panel['active_conflict_age'] = df_panel['active_conflict_age'].where(
    df_panel['active_conflict_age'] >= 0
)

# Total and active duration
df_panel['duration_months'] = (
    df_panel.groupby('conflict_id')['end_date_numeric'].transform('max') -
    df_panel.groupby('conflict_id')['start_date_numeric'].transform('min')
)
df_panel['active_duration_months'] = (
    df_panel.groupby('conflict_id')['ged_event_month'].transform('sum')
)

# Calendar features
df_panel['year_mo']       = df_panel['year_mo'].astype(str)
df_panel['year']          = df_panel['year_mo'].str[:4].astype(int)
df_panel['start_year']    = df_panel['start_date'].astype(str).str[:4].astype(int)
df_panel['current_month'] = pd.to_datetime(df_panel['year_mo']).dt.month

---
## 9. GDP per capita

In [44]:
gdp_pc = pd.read_csv('../../data/input/gdp_pc/gdp_pc.csv')
gdp_pc['year_mo'] = (
    pd.to_datetime(gdp_pc['year_mo']).dt.to_period('M').dt.to_timestamp()
    .astype(str).str[:7]
)
df_panel = df_panel.merge(gdp_pc, on=['isocode', 'year_mo'], how='left')

---
## 10. Treatment variables

Two parallel sets of treatment variables are built:

| Variable | Definition | Count | Use |
|---|---|---|---|
| `first_agreement_date` | Month index of first PA-X agreement (any month) | — | Survival analysis |
| `first_agreement_ged_date` | Month index of first PA-X agreement in GED-event month | — | CSDID |
| `ever_agreement` | 1 if conflict signed any PA-X agreement | 103 conflicts | Survival analysis |
| `ever_agreement_ged` | 1 if conflict signed during a GED-event month | 71 conflicts | CSDID |

In [45]:
new_cols = {}

for col in agreement_base_cols:
    # Any-month definition (for survival analysis)
    first_any = (
        df_panel.loc[df_panel[col] == 1]
        .groupby('conflict_id')['year_mo_numeric'].min()
    )
    new_cols[f'first_{col}_date'] = (
        df_panel['conflict_id'].map(first_any).fillna(0).astype(int)
    )
    new_cols[f'ever_{col}'] = (
        df_panel.groupby('conflict_id')[col].transform('max').astype(int)
    )

    # GED-event-month definition (for CSDID)
    ged_col = f'{col}_ged'
    first_ged = (
        df_panel.loc[df_panel[ged_col] == 1]
        .groupby('conflict_id')['year_mo_numeric'].min()
    )
    new_cols[f'first_{col}_ged_date'] = (
        df_panel['conflict_id'].map(first_ged).fillna(0).astype(int)
    )
    new_cols[f'ever_{col}_ged'] = (
        df_panel.groupby('conflict_id')[ged_col].transform('max').astype(int)
    )

df_panel = pd.concat([df_panel, pd.DataFrame(new_cols)], axis=1)

n_total = df_panel['conflict_id'].nunique()
n_any   = df_panel.groupby('conflict_id')['ever_agreement'].max().sum()
n_ged   = df_panel.groupby('conflict_id')['ever_agreement_ged'].max().sum()

print(f"Total conflicts: {n_total}")
print(f"  ever_agreement     (any month)         : {int(n_any):>3}  → for survival analysis")
print(f"  ever_agreement_ged (GED-event month)   : {int(n_ged):>3}  → for CSDID")
print(f"  Difference (agreements in dormancy/post-GED only): {int(n_any - n_ged)}")

Total conflicts: 201
  ever_agreement     (any month)         : 104  → for survival analysis
  ever_agreement_ged (GED-event month)   :  71  → for CSDID
  Difference (agreements in dormancy/post-GED only): 33


---
## 11. Number of armed factions (dyads)

In [46]:
def count_factions(x):
    if isinstance(x, (list, np.ndarray)):
        return len(x)
    if pd.isna(x):
        return np.nan
    if isinstance(x, str):
        return len(x.split())
    return np.nan

df_panel['n_factions'] = df_panel['dyad_id'].apply(count_factions)

---
## 12. Export

Exports `conflict_panel.csv` to `data/output/conflict_level/`.

In [ ]:
df_panel.columns = (
    df_panel.columns
    .str.replace(' ', '_')
    .str.replace('-', '_')
    .str.replace('.', '_')
)
df_panel['conflict_id'] = df_panel['conflict_id'].astype(int)
df_panel['best']        = df_panel['best'].astype(float)
df_panel['log_best']    = df_panel['log_best'].astype(float)

df_panel.to_csv('../../data/output/conflict_level/conflict_panel.csv', index=False)

print("Exported: conflict_panel.csv")
print(f"  Rows      : {len(df_panel):,}")
print(f"  Conflicts : {df_panel['conflict_id'].nunique()}")
print(f"  Columns   : {df_panel.shape[1]}")
print()
print("Key agreement column summary:")
print(f"  agreement          (any month)        : "
      f"{df_panel.groupby('conflict_id')['ever_agreement'].max().sum():.0f} treated conflicts")
print(f"  agreement_ged      (GED-event month)  : "
      f"{df_panel.groupby('conflict_id')['ever_agreement_ged'].max().sum():.0f} treated conflicts")

Exported: conflict_panel_unified.csv
  Rows      : 86,832
  Conflicts : 201
  Columns   : 147

Key agreement column summary:
  agreement          (any month)        : 104 treated conflicts
  agreement_ged      (GED-event month)  : 71 treated conflicts
